In [2]:
%pip install -Uq "unstructured[all-docs]" 
%pip install -Uq langchain_chroma 
%pip install -Uq langchain langchain-community langchain-google-genai 
%pip install -Uq python_dotenv
%pip install -Uq langchain-huggingface

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import json
from typing import List

from unstructured.partition.pdf import partition_pdf
from unstructured.chunking.title import chunk_by_title

from langchain_core.documents import Document
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_core.messages import HumanMessage
from dotenv import load_dotenv

load_dotenv()

import os
# Add the exact path to your Tesseract installation
os.environ["PATH"] += os.pathsep + r'C:\Program Files\Tesseract-OCR'

e:\Training Resources\Retrieval-Augmented-Generation\Multi-Modal RAG\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def partition_document(file_path: str):
  """Converting all whole document into individual small Atom elements"""
  print(f"📄 Partitioning Document: {file_path}")

  elements = partition_pdf(
    filename=file_path,
    strategy="hi_res", # Use the most accurate (but slower) processing method of extraction
    infer_table_structure=True, # Keep tables as structured HTML, not jumbled text
    extract_image_block_types=["Image"], # Grab images found in the PDF
    extract_image_block_to_payload=True # Store images as base64 data you can actually use
  )

  print(f"✅ Successfully Extracted {len(elements)} elements from {file_path} document!")
  return elements

file_path = "./sources/LatexSource_184.pdf"
elements = partition_document(file_path)

📄 Partitioning Document: ./sources/LatexSource_184.pdf


Loading weights: 100%|██████████| 367/367 [00:00<00:00, 3260.50it/s]


✅ Successfully Extracted 182 elements from ./sources/LatexSource_184.pdf document!


In [3]:
elements

In [4]:
# All the types of unique atom elements that we get from the unstructured.io
set([str(type(el)) for el in elements])

{"<class 'unstructured.documents.elements.FigureCaption'>",
 "<class 'unstructured.documents.elements.Header'>",
 "<class 'unstructured.documents.elements.Image'>",
 "<class 'unstructured.documents.elements.ListItem'>",
 "<class 'unstructured.documents.elements.NarrativeText'>",
 "<class 'unstructured.documents.elements.Table'>",
 "<class 'unstructured.documents.elements.Text'>",
 "<class 'unstructured.documents.elements.Title'>"}

In [5]:
elements[0].to_dict()

{'type': 'Title',
 'element_id': '488a9f7156ce65ab638193323c24346c',
 'text': 'Data-Driven Rockfall Assessment using Synthetic Training and CNN–XGBoost Integration',
 'metadata': {'detection_class_prob': 0.8782138824462891,
  'is_extracted': 'true',
  'coordinates': {'points': ((np.float64(645.2001953125),
     np.float64(555.3088469444441)),
    (np.float64(645.2001953125), np.float64(713.6800537109375)),
    (np.float64(2339.795166015625), np.float64(713.6800537109375)),
    (np.float64(2339.795166015625), np.float64(555.3088469444441))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-02-22T17:17:51',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 1,
  'file_directory': './sources',
  'filename': 'LatexSource_184.pdf'}}

In [6]:
# Get all the images
images = [element for element in elements if element.category=="Image"]
print(f"🖼️ Total {len(images)} images found in the {file_path} document!")

images[4].to_dict()

# Convert the base64 to actual image at: https://base64.guru/converter/decode/image

🖼️ Total 5 images found in the ./sources/LatexSource_184.pdf document!


{'type': 'Image',
 'element_id': '1f9e51943e493645ba3a96a1bb8460f9',
 'text': 'XGBoost Feature Importance CrackScore SlopeAngle Rainfall Curvature Strain tures o Displacement PorePressure Temperature Vibrations Aspect 0.0 0.1 0.2 0.3 Importance Score 0.4',
 'metadata': {'coordinates': {'points': ((np.float64(1462.0277777777776),
     np.float64(564.4350094444447)),
    (np.float64(1462.0277777777776), np.float64(1186.4319444444445)),
    (np.float64(2336.2196277777775), np.float64(1186.4319444444445)),
    (np.float64(2336.2196277777775), np.float64(564.4350094444447))),
   'system': 'PixelSpace',
   'layout_width': 2975,
   'layout_height': 3850},
  'last_modified': '2026-02-22T17:17:51',
  'filetype': 'application/pdf',
  'languages': ['eng'],
  'page_number': 11,
  'image_base64': '...',
  'image_mime_type': 'image/jpeg',
  'file_directory': './sources',
  'filename': 'LatexSource_184.pdf'}}

In [7]:
# Get all the tables
tables = [element for element in elements if element.category=="Table"]
print(f"📊 Total {len(tables)} tables found in the {file_path} document!")

tables[0].to_dict()

📊 Total 6 tables found in the ./sources/LatexSource_184.pdf document!


{'type': 'Table',
 'element_id': 'ff33f476fbb95bb7770126a9af25d4af',
 'text': 'Data Primary Goal Sample Size Features Category Historical Rainfall- 11,610 landslides Elevation, slope, aspect, records from induced (FR), 256 dated lithology, land cover, NDVI, Yunnan landslide (train/test), 20 soil type, average annual Province, forecasting and dated (validation), precipitation, event daily China. [17] susceptibility matched rain, antecedent effective mapping [17] non-landslide rainfall [17] samples [17] Historical Regional 57 rockfall events; Precipitation, slope aspect, inventory and rockfall 157 labelled units slope angle, weathering, field surveys in susceptibility (57 susceptible, 100 lithology, distance to rivers, Alborz, Iran. with ANN, non-susceptible), faults unsafe radius, [25] compared to 70/30 split [25] landslide-prone areas, SVM/DT/RF distance to roads, distance to [25] cities, RQD, Qslope, GSI, SMR, RHRS [25] 5-year 3D Automated 8,966 deformation Raw 3D point coordinates, L

In [9]:
def create_chunks_by_title(elements):
  print("🔨 Creating intelligent chunks using the title-based strategy")

  chunks = chunk_by_title(
    elements=elements,
    max_characters=3000, # Hard limit - never exceed 3000 characters per chunk
    new_after_n_chars=2400, # Try to start a new chunk after 2400 characters
    combine_text_under_n_chars=600 # Merge tiny chunks under 500 chars with neighbors
  )

  print(f"✅ Successfully created {len(chunks)} chunks!")
  return chunks

chunks = create_chunks_by_title(elements)

🔨 Creating intelligent chunks using the title-based strategy
✅ Successfully created 21 chunks!


In [45]:
# Get all unique chunks
set([str(type(chunk)) for chunk in chunks])

{"<class 'unstructured.documents.elements.CompositeElement'>",
 "<class 'unstructured.documents.elements.Table'>"}

In [10]:
chunks[10].metadata.orig_elements

chunks[10].metadata.orig_elements[0].to_dict()

{'type': 'Image',
 'element_id': '0412f75a3377bf66eff0835830525a57',
 'text': "Feature Blurprint 10 input features + 1 target Scenario 1: Intense Rainfall Scenario 2: Vibration-Induced Rainfall: 100-200mm_ Vibrations: 20-50 mms PorePressure: 70-100 kPa “CrackScore: 0.7-1.0) CrackScore: 0.6-0.9 'SlopeAngle: 70°-85°, SlopeAngle: 60°-80° ‘Rainfall: Low) Temperature: 15°-30° SlopeAngle | |Aspect} | Strain _PorePressure: Low | Scenario 3: Freezing Temp. Temperature: -5°C to +5°C _CrackScore: 0.8-1.0 Displacement: 2-6 mmiweek) “Rainfall: Low ‘PorePressure: 10-30kPa_ (Target: High Risk (1)! |Target: High Risk (1)! Target: High Risk (1)| m> { 1 Low Risk Scenario x Curvature | | Displacement , ae . Scenario 4: Combination Scenario 5: Progressive Scenario 3: All Clear Rainfall Failure . Failure “Temperature: -5°C to 45°C. Rainfall: 40-80 i - 8. f aintal mm) ; Displacement: 8-20 mm/week CrackScore: 0.0-0.2 PorePressure: 30-60 kPa in f ; ’ || | Strain: 3000-5000 pre Displacement: <0.5 mm/week Crac

In [11]:
chunks[0].metadata.orig_elements[0].to_dict()["type"]

'Title'

In [13]:
def separate_content_types(chunk):
  """Analyze and separate different types of elements in the chunk"""
  content_data = {
    'text': chunk.text,
    'tables': [],
    'images': [],
    'types': ['text']
  }

  if hasattr(chunk, 'metadata') and hasattr(chunk.metadata, 'orig_elements'):
    for element in chunk.metadata.orig_elements:
      element_dict = element.to_dict()
      element_type = element_dict['type']

      if element_type=="Table":
        content_data['types'].append('table')
        table_html = element_dict['metadata']['text_as_html']
        content_data['tables'].append(table_html)

      if element_type=="Image":
        if hasattr(element, 'metadata') and hasattr(element.metadata, 'image_base64'):
          content_data['types'].append('image')
          content_data['images'].append(element_dict['metadata']['image_base64'])
    
  content_data['types'] = list(set(content_data['types']))
  return content_data

In [14]:
def create_ai_enhanced_summary(text:str, tables: List[str], images: List[str]) -> str:
  """Create AI enhanced summary for composite content"""

  try:
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0.2)

    prompt = f"""You are creating a searchable description for document content retrieval.

    CONTENT TO ANALYZE:
    TEXT CONTENT:
    {text}

    """

    if tables:
      prompt += "TABLES:\n"
      for i, table in enumerate(tables):
        prompt += f"Table {i+1}:\n{table}\n\n"
        prompt += """
        YOUR TASK:
        Generate a comprehensive, searchable description that covers:

        1. Key facts, numbers, and data points from text and tables
        2. Main topics and concepts discussed  
        3. Questions this content could answer
        4. Visual content analysis (charts, diagrams, patterns in images)
        5. Alternative search terms users might use

        Make it detailed and searchable - prioritize findability over brevity.

        SEARCHABLE DESCRIPTION:"""

    message_content = [{"type": "text", "text": prompt}]

    if images:
      for images_base64 in images:
        message_content.append({
          "type": "image_url",
          "image_url": {"url": f"data:image/jpeg;base64,{images_base64}"}
        })

    message = HumanMessage(content=message_content)
    response = llm.invoke([message])

    return response.content
  except Exception as e:
    print(f"    ❌ AI summary failed: {e}")
    summary = f"{text[:300]}..."
    if tables:
      summary += f" [Contains {len(tables)} tables]"
    if images:
      summary += f" [Contains {len(images)} images]"
    return summary

In [15]:
def summarise_chunk(chunks):
  """Process all chunks with AI Summary"""
  print("🧠 Processing chunks with AI Summaries...")

  langchain_documents = []
  total_chunks = len(chunks)

  for i, chunk in enumerate(chunks):
    current_chunk = i+1
    print(f"⌛ Processing chunk {current_chunk}/{total_chunks}...")

    content_data = separate_content_types(chunk)

    print(f"    Total types of elemets found: {len(content_data)}")
    print(f"    Tables: {len(content_data['tables'])}, Images: {len(content_data['images'])}")

    if content_data['tables'] or content_data['images']:
      print(f"    => Creating AI enhanced summary for mixed content...")
      try:
        enhanced_content = create_ai_enhanced_summary(
          content_data['text'],
          content_data['tables'],
          content_data['images']
        )
        print(f"    => AI enhanced summary created successfull!")
        print(f"    => Enhanced content preview: {enhanced_content[:200]}...")
      except Exception as e:
        print(f"    => Failed to create AI Summary: {e}")
        enhanced_content = content_data['text']
    else:
      print(f"    => Using raw text (no images/no tables)")
      enhanced_content = content_data['text']

    langchain_doc = Document(
      page_content=enhanced_content,
      metadata={
        "original_content": json.dumps({
          "raw_text": content_data['text'],
          "tables_html": content_data['tables'],
          "images_base64": content_data['images']
        })
      }
    )

    langchain_documents.append(langchain_doc)

  print(f"✅ Processed {len(langchain_documents)} chunks.")
  return langchain_documents

processed_chunks = summarise_chunk(chunks)
    

🧠 Processing chunks with AI Summaries...
⌛ Processing chunk 1/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 2/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 3/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 4/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 5/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 6/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 7/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 8/21...
    Tota

In [16]:
processed_chunks[0]

Document(metadata={'original_content': '{"raw_text": "Data-Driven Rockfall Assessment using Synthetic Training and CNN\\u2013XGBoost Integration\\n\\nManas Sunil Patil1[0009\\u22120005\\u22121116\\u22123213] and Dr. Vishvajit\\n\\nBakarola2[0000\\u22120002\\u22126602\\u22129194]\\n\\n1 Asha M. Tarsadia Institute of Computer Science and Technology, Uka Tarsadia\\n\\nUniversity, Bardoli, Gujarat, 394350, India\\n\\n22amtics235@gmail.com\\n\\n2 Asha M. Tarsadia Institute of Computer Science and Technology, Uka Tarsadia University, Bardoli, Gujarat, 394350, India\\n\\nvishvajit.bakrola@utu.ac.in\\n\\nAbstract. Rockfall poses a persistent risk to life and infrastructure,\\n\\ndemanding proactive, site-specific forecasting. We propose a Hybrid AI framework that fuses convolutional neural networks for quantitative anal- ysis of rock-surface imagery with an XGBoost model ingesting geospatial covariates like rainfall, pore pressure, and temperature to estimate an early-warning probability of sl

In [19]:
def export_chunk_to_json(processed_chunks, filename="processed_chunks_export.json"):
  """Export processed chunks to clean JSON format"""
  print(f"⌛ Exporting the processed chunks into clean JSON format at {filename}")
  export_data = []

  for i, doc in enumerate(processed_chunks):
    chunk_data = {
      "chunk_id": i+1,
      "enhanced_content": doc.page_content,
      "metadata": {
        "original_content": json.loads(doc.metadata.get("original_content", "{}"))
      }
    }

    export_data.append(chunk_data)

  with open(filename, "w", encoding='utf-8') as file:
    json.dump(export_data, file, indent=2, ensure_ascii=True)

  print(f"✅ Exported {len(export_data)} chunks to {filename}")
  return export_data
  
json_data = export_chunk_to_json(processed_chunks)

⌛ Exporting the processed chunks into clean JSON format at processed_chunks_export.json
✅ Exported 21 chunks to processed_chunks_export.json


In [22]:
def create_vector_store(documents, persist_directory="dbv1/chroma_db"):
  """Create and persist ChromaDB vector store"""
  print(f"⌛ Creating embeddings and storing in ChromaDB...")

  embedding_model = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")

  vector_store = Chroma.from_documents(
    documents=documents,
    persist_directory=persist_directory,
    embedding=embedding_model,
    collection_metadata={"hnsw:space": "cosine"}
  )

  print(f"✅ Vector store created successfully and saved at {persist_directory}")
  return vector_store

vector_store = create_vector_store(processed_chunks)

⌛ Creating embeddings and storing in ChromaDB...


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5122.77it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector store created successfully and saved at dbv1/chroma_db


In [25]:
query = "What are the main components of the Hybrid Pipeline used in this reasearch?"
retriever = vector_store.as_retriever(search_kwargs={"k": 3})
retrieved_chunks = retriever.invoke(query)

retrieved_chunks_as_json = export_chunk_to_json(retrieved_chunks, "rag_results.json")

⌛ Exporting the processed chunks into clean JSON format at rag_results.json
✅ Exported 3 chunks to rag_results.json


In [29]:
def run_complete_ingestion_pipeline(pdf_path: str):
  """Run the complete RAG ingestion pipeline"""
  print(f"🚀 Running the complete RAG ingestion pipeline...")
  print("="*50)

  elements = partition_document(pdf_path)

  chunks_by_title = create_chunks_by_title(elements)

  processed_chunks = summarise_chunk(chunks_by_title)

  vector_db = create_vector_store(processed_chunks)

  print(f"✅ Pipeline execution completed successfully!")
  return vector_db

In [ ]:
db = run_complete_ingestion_pipeline("./sources/LatexSource_184.pdf")

🚀 Running the complete RAG ingestion pipeline...
📄 Partitioning Document: ./sources/LatexSource_184.pdf
✅ Successfully Extracted 182 elements from ./sources/LatexSource_184.pdf document!
🔨 Creating intelligent chunks using the title-based strategy
✅ Successfully created 21 chunks!
🧠 Processing chunks with AI Summaries...
⌛ Processing chunk 1/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 2/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 3/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 4/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chunk 5/21...
    Total types of elemets found: 4
    Tables: 0, Images: 0
    => Using raw text (no images/no tables)
⌛ Processing chun

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7923.65it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Vector store created successfully and saved at dbv1/chroma_db
✅ Pipeline execution completed successfully!


In [42]:
query = "What is the hardware configuration used in this research paper?"

retriever = db.as_retriever(search_kwarg={"k": 3})
searched_chunks = retriever.invoke(query)

def generate_final_answer(chunks, query):
  """Generate final answer using multimodal contents"""

  try:
    llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash-lite", temperature=0.2)

    prompt = f"""Based on the following documents, please answer this question: {query}

    CONTENT TO ANALYZE:
    """

    for i, doc in enumerate(chunks):
      prompt += f"- Document {i+1}\n"
      if "original_content" in doc.metadata:
        original_content = json.loads(doc.metadata["original_content"])

        raw_text = original_content.get("raw_text", "")
        if raw_text:
          prompt += f"TEXT:\n{raw_text}\n\n"

        tables_html = original_content.get("tables_html", [])
        if tables_html:
          prompt += f"TABLES:\n"
          for j, table_html in enumerate(tables_html):
            prompt += f"-- Table {i+1}:\n{table_html}\n\n"

    prompt += f"""
    Please provide a clear, comprehensive answer using the text, tables, and images above. If the documents don't contain sufficient information to answer the question, say "I don't have enough information to answer that question based on the provided documents.

    ANSWER:
    """

    message_content = [{"type": "text", "text": prompt}]

    for doc in chunks:
      if "original_content" in doc.metadata:
        original_content = json.loads(doc.metadata["original_content"])
        images_base64 = original_content.get("images_base64")

        if images_base64:
          for image_base64 in images_base64:
            message_content.append({
              "type": "image_url",
              "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"}
            })

    message = HumanMessage(content=message_content)
    response = llm.invoke([message])

    return response.content
  except Exception as e:
    print(f"❌ Answer generation failed: {e}")
    return "Sorry, I encountered an error while generating an answer!"
  
final_answer = generate_final_answer(searched_chunks, query)
print(final_answer)

The hardware configuration used in this research paper includes an **NVIDIA GeForce RTX 3050 GPU**.
